https://github.com/ijstokes/tutorial-local-llm

---

# Local LLM

---

In [5]:
# load in libraries
import json
from ollama import Client, chat, list as list_models
from pathlib import Path
import pandas as pd
import subprocess
import time


In [25]:
# Start the Ollama server if it is not already running
ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give the server a moment to start
time.sleep(3)

# Check whether the server is running
print("Ollama process:", ollama_server.poll())

Ollama process: None


In [26]:
# Check which models are available
models = list_models()

print(models)

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 57, 137259, tzinfo=TzInfo(39600)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')), Model(model='codellama:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 51, 300149, tzinfo=TzInfo(39600)), digest='8fdf8f752f6e80de33e82f381aba784c025982752cd1ae9377add66449d2225f', size=3825910662, details=ModelDetails(parent_model='', format='gguf', family='llama', families=None, parameter_size='7B', quantization_level='Q4_0')), Model(model='mistral:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 46, 29050, tzinfo=TzInfo(39600)), digest='6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf', size=4372824384, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], 

In [27]:
CONDITION = "aig"

# Set the path to the processed passages
DATA_DIR = Path("..") / "pmc" / CONDITION / "processed"

### Full-text Articles

In [66]:
# Load all full-text passages
all_passages = pd.read_parquet(
    DATA_DIR / "passages.parquet"
)

# Group all passages by article
fulltext_articles = (
    all_passages.sort_values(["pmcid", "passage_id"])
    .groupby("pmcid", as_index=False)
    .agg(
        text=("text", lambda x: "\n\n".join(x.dropna())),
        passage_ids=("passage_id", list),
        n_passages=("passage_id", "size")
    )
)

# Check the number of articles and passages
print("Articles:", len(fulltext_articles))
print("Passages:", fulltext_articles["n_passages"].sum())

display(fulltext_articles.head())

Articles: 152
Passages: 4131


,pmcid,text,passage_ids,n_passages
0,PMC10052490,We report an unusual case of autoimmune gastri...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]",11
1,PMC10076145,Autoimmune gastritis (AIG) typically exhibits ...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",35
2,PMC10241230,SummaryAutoimmune polyendocrinopathy-candidias...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",29
3,PMC10258108,Epstein-Barr virus-associated gastric cancer (...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",19
4,PMC10290196,Key Clinical MessageWe describe an ultra‐early...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",38


### MedCPT

In [30]:
# Load the diagnostic passages grouped by article
diagnostic_articles = pd.read_parquet(
    DATA_DIR / "diagnostic_articles.parquet"
)

# Check the number of articles and passages
print("Articles:", len(diagnostic_articles))
print("Passages:", diagnostic_articles["n_passages"].sum())

display(diagnostic_articles)

Articles: 5
Passages: 10


,pmcid,text,passage_ids,n_passages,max_score
0,PMC11010983,The prevalence of AAG in the general populatio...,"[5, 17]",2,69.326569
1,PMC11863545,The objective of this retrospective cohort stu...,[6],1,69.230820
2,PMC3520153,The cut-off values adopted were those suggeste...,[8],1,69.347687
3,PMC6502205,"In practical terms, it is advisable to screen ...",[16],1,69.181526
4,PMC8621657,CAG can be asymptomatic or can be characterize...,"[9, 20, 26, 27, 28]",5,70.686111


### BERTopic

In [64]:
# Get the unique PMC IDs
selected_pmcids = diagnostic_articles["pmcid"].unique().tolist()

print("Selected articles:", len(selected_pmcids))
print(selected_pmcids)

Selected articles: 5
['PMC11010983', 'PMC11863545', 'PMC3520153', 'PMC6502205', 'PMC8621657']


In [67]:
# Select the same articles used in MedCPT
fulltext_articles = fulltext_articles[
    fulltext_articles["pmcid"].isin(selected_pmcids)
].copy()

# Check the selected articles
print("Selected articles:", len(fulltext_articles))
display(fulltext_articles[["pmcid", "n_passages"]])

Selected articles: 5


,pmcid,n_passages
20,PMC11010983,43
34,PMC11863545,29
67,PMC3520153,21
94,PMC6502205,16
124,PMC8621657,40


## Header

In [68]:
# Connect to the local Ollama server
client = Client(host="http://localhost:11434")

# select the local llm to use
#MODEL = "llama3.2"
MODEL = "mistral"

In [50]:
# Create instructions for extracting diagnostic criteria
PROMPT = """
Extract diagnostic criteria for autoimmune gastritis (AIG)
from the provided medical passages.

Only extract information explicitly stated in the source text.
Do not invent diagnostic criteria, tests or thresholds.

Do not extract symptoms, complications, prevalence statistics
or screening recommendations unless explicitly described
as diagnostic criteria.

Return a JSON object containing a list named "criteria".

For each diagnostic criterion, extract:
- condition: the disease being diagnosed
- criterion: description of the diagnostic criterion
- category: serological, histological, endoscopic, clinical,
  microbiological, imaging or other
- criteria_type: general diagnostic criterion, study-specific
  diagnostic criterion or unspecified
- diagnostic_test: test used to identify the criterion
- marker: biomarker or characteristic being assessed
- threshold: numerical cut-off including comparison operator
- unit: measurement unit
- diagnostic_role: confirmatory, supportive, exclusionary
  or unspecified
- combination: other diagnostic criteria required in combination
- evidence: exact supporting quotation from the source text

Follow these rules:
- Preserve numerical thresholds and their measurement units.
- Preserve AND/OR relationships between diagnostic criteria.
- Do not assume that individual findings confirm a diagnosis.
- Ensure the evidence directly supports the extracted criterion.
- Copy evidence verbatim without paraphrasing.
- Use null for missing information, not the string "None".
- Only provide units for numerical measurements.

Return valid JSON only, without Markdown or commentary.

If no diagnostic criteria are found, return:
{{"criteria": []}}

Passages:
{passage}
"""

In [69]:
prompt = PROMPT

In [70]:
# Create a function to send prompts to the local LLM
def run_llm(prompt, model=MODEL):

    # Send the prompt to the local LLM
    response = chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0,
            "num_ctx": 16384
        }
    )

    # Return the generated response
    return response.message.content, response.done_reason

In [71]:
# Create a function to extract diagnostic criteria from articles
def extract_articles(articles_df, method, model=MODEL):

    # Create a list to store the extraction results
    results = []

    # Go through each article
    for _, row in articles_df.iterrows():

        PMCID = row["pmcid"]
        passage = row["text"]

        # Insert the article text into the diagnostic prompt
        prompt = PROMPT.format(passage=passage)

        # Start measuring LLM wall-clock time
        start = time.perf_counter()

        # Extract diagnostic criteria
        result = run_llm(prompt, model=model)

        # Stop measuring LLM wall-clock time
        llm_time = time.perf_counter() - start

        # Store the extraction results
        results.append({
            "pmcid": PMCID,
            "model": model,
            "retrieval_method": method,
            "n_passages": int(row["n_passages"]),
            "passage_ids": [int(pid) for pid in row["passage_ids"]],
            "llm_time_seconds": llm_time,
            "response": result
        })

        print(f"Processed: {PMCID} ({llm_time:.2f} seconds)")

    return results

In [72]:
fulltext_results = extract_articles(
    fulltext_articles,
    method="full_text"
)

Processed: PMC11010983 (10.62 seconds)
Processed: PMC11863545 (5.41 seconds)
Processed: PMC3520153 (5.97 seconds)
Processed: PMC6502205 (7.95 seconds)
Processed: PMC8621657 (9.12 seconds)


In [75]:
# Calculate total LLM time
total_llm_time = sum(
    result["llm_time_seconds"]
    for result in fulltext_results
)

print(f"Total LLM time: {total_llm_time:.2f} seconds")

Total LLM time: 39.07 seconds


In [56]:
medcpt_results = extract_articles(
    diagnostic_articles,
    method="medcpt"
)

Processed: PMC11010983 (4.68 seconds)
Processed: PMC11863545 (3.71 seconds)
Processed: PMC3520153 (1.33 seconds)
Processed: PMC6502205 (3.51 seconds)
Processed: PMC8621657 (3.43 seconds)


In [74]:
# Calculate total LLM time
total_llm_time_medcpt = sum(
    result["llm_time_seconds"]
    for result in medcpt_results
)

print(f"Total LLM time: {total_llm_time_medcpt:.2f} seconds")

Total LLM time: 16.66 seconds


In [76]:
# Create a function to save the extraction results as JSON
def save_results(results, method, output_dir=DATA_DIR):

    # Convert results into a JSON file
    with open(
        output_dir / f"{method}_llm_results.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

In [77]:
# Save the extraction results for each model and pipeline
model_name = MODEL.replace(".", "")

save_results(medcpt_results, f"medcpt_{model_name}")
save_results(fulltext_results, f"fulltext_{model_name}")

In [78]:
# Create a function to display the LLM extraction results
def display_results(results):

    # Convert the results into a DataFrame
    results_df = pd.DataFrame(results)

    display(results_df)

    # Display the extracted diagnostic criteria for each article
    for result in results:

        print("Article:", result["pmcid"])

        # Get the generated text from the LLM response
        response = result["response"]

        # Handle responses containing text and completion reason
        if isinstance(response, (list, tuple)):
            response = response[0]

        # Display the response
        print(response)
        print("-" * 80)

    return results_df

In [79]:
fulltext_df = display_results(fulltext_results)

,pmcid,model,retrieval_method,n_passages,passage_ids,llm_time_seconds,response
0,PMC11010983,mistral,full_text,43,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",10.621043,"( {\n ""criteria"": [\n {\n ""condition""..."
1,PMC11863545,mistral,full_text,29,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",5.414246,"( {\n ""criteria"": [\n {\n ""condition""..."
2,PMC3520153,mistral,full_text,21,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",5.969847,"( {\n ""criteria"": [\n {\n ""condition""..."
3,PMC6502205,mistral,full_text,16,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",7.950539,"( {\n ""criteria"": [\n {\n ""condition""..."
4,PMC8621657,mistral,full_text,40,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",9.116678,"( {\n ""criteria"": [\n {\n ""condition""..."


Article: PMC11010983
 {
  "criteria": [
    {
      "condition": "Autoimmune atrophic gastritis",
      "criterion": "Presence of atrophy in the oxyntic mucosa due to anti-parietal cell antibodies",
      "category": "Histological",
      "criteria_type": "General diagnostic criterion",
      "diagnostic_test": "Histological examination of gastric biopsies",
      "marker": "Atrophy in the oxyntic mucosa",
      "threshold": null,
      "unit": null,
      "diagnostic_role": "Confirmatory",
      "combination": null,
      "evidence": "Patients with AAG are often asymptomatic but can also have dyspeptic or reflux symptoms. The atrophy of the oxyntic mucosa leads to iron and vitamin B12 malabsorption, which may result in anemia and neurological affections."
    },
    {
      "condition": "Autoimmune atrophic gastritis",
      "criterion": "Presence of anti-parietal cell antibodies",
      "category": "Serological",
      "criteria_type": "General diagnostic criterion",
      "diagnosti

In [63]:
medcpt_df = display_results(medcpt_results)

,pmcid,model,retrieval_method,n_passages,passage_ids,llm_time_seconds,response
0,PMC11010983,mistral,medcpt,2,"[5, 17]",4.684541,"( {\n ""criteria"": [\n {\n ""condition""..."
1,PMC11863545,mistral,medcpt,1,[6],3.705679,"( {\n ""criteria"": [\n {\n ""condition""..."
2,PMC3520153,mistral,medcpt,1,[8],1.327513,"( {\n ""criteria"": [\n {\n ""condition""..."
3,PMC6502205,mistral,medcpt,1,[16],3.512888,"( {\n ""criteria"": [\n {\n ""condition""..."
4,PMC8621657,mistral,medcpt,5,"[9, 20, 26, 27, 28]",3.430286,"( {\n ""criteria"": [\n {\n ""condition""..."


Article: PMC11010983
 {
  "criteria": [
    {
      "condition": "Autoimmune gastritis (AIG)",
      "criterion": "Histological examination of gastric biopsy specimens for the diagnosis of AIG",
      "category": "histological",
      "criteria_type": "general diagnostic criterion",
      "diagnostic_test": "Histological examination",
      "marker": "Presence of atrophic gastritis",
      "threshold": null,
      "unit": null,
      "diagnostic_role": "confirmatory",
      "combination": null,
      "evidence": "Some studies correctly used histological examination of gastric biopsy specimens for the diagnosis of AAG."
    },
    {
      "condition": "Autoimmune gastritis (AIG)",
      "criterion": "Endoscopic findings in AAG include mucosal erythema, nodularity, and an ipo-atrophic mucosal pattern",
      "category": "endoscopic",
      "criteria_type": "general diagnostic criterion",
      "diagnostic_test": "Endoscopic examination",
      "marker": "Presence of mucosal erythema, nod

In [ ]:
#bertopic_df = display_results(bertopic_results)

In [24]:
# Stop the Ollama server
ollama_server.terminate()
ollama_server.wait()


0